# Methods

For project 2, I started by creating a baseline model using SVR. In this baseline model, I included a validation set so that I could get immediate results from experiments to conserve submissions. In order to make the baseline model run properly, I added some preprocessing to the parameters in the training, validation, and test sets. The preprocessing I added included one-hot encoding all of the categorical variables with get_dummies (including majors), so that the variables are able to be processed in the model numerically. In terms of dealing with missing data, I replaced missing data with the mean instead of with zeros so that there is a reduction in outliers. In terms of model parameters, I received the highest score when I had SVR with only the "linear" parameter. I did perform a GridSearchCV to search for the most optimal SVR parameters, but the search resulted in a higher MSE. SVR worked incrediby well, so instead of testing different models, I focused on testing SVR's parameters as well as the preprocessing. Another preprocessing experiment I did was with PCA (it worsened the score strongly),Cross Validation (helped with analyzing performance) and LassoCV to possibly remove unnecessary paramters (this did recduce overfitting). This final submission is the version with the LassoCV unnecessary feature removals and baseline SVR model. 

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/project-2-cosc-220-spring-2025/train.csv
/kaggle/input/project-2-cosc-220-spring-2025/test.csv
/kaggle/input/project-2-cosc-220-spring-2025/archive/train.json
/kaggle/input/project-2-cosc-220-spring-2025/archive/test.json


In [2]:
#Train + Validation Data Setup

file_path = '/kaggle/input/project-2-cosc-220-spring-2025/train.csv'
TrainData = pd.read_csv(file_path)
#checking to make sure the data was imported correctly
print(TrainData.head())

#Making the validation set
from sklearn.model_selection import train_test_split
TrainData, ValData = train_test_split(TrainData, test_size=0.2, shuffle=True, random_state=42)


#Changes the categorical variables to number format
#get_dummies = does one hot encoding on the values in the dataset
TrainData = pd.get_dummies(TrainData, columns=['Gender', 'Admit Type', 'Major'], drop_first=True)

#____________________________________________________________________________________________________________________
#EXPERIMENT: How to deal with zero values in the data
# 2 options: delete zero data columns, or replace with the mean. 
#I went with the mean option
TrainData.fillna(TrainData.mean(), inplace=True)
#____________________________________________________________________________________________________________________

#Changes the categorical variables to number format
#get_dummies = does one hot encoding on the values in the dataset
ValData = pd.get_dummies(ValData, columns=['Gender', 'Admit Type', 'Major'], drop_first=True)
#ChatGPT recommended adding this to check that the columns in ValData match the columns in TrainData
ValData = ValData.reindex(columns=TrainData.columns, fill_value=0)

#____________________________________________________________________________________________________________________
#EXPERIMENT: How to deal with zero values in the data
# 2 options: delete zero data columns, or replace with the mean. 
#I went with the mean option
ValData.fillna(ValData.mean(), inplace=True)
#____________________________________________________________________________________________________________________


#checking to make sure everything looks right
print(f'Train Data Size: {len(TrainData)}')
print(f'Validation Data Size: {len(ValData)}')

#Target value setup
target_value = 'GPA'
#this makes sure the value GPA maps to the actual data instead of being a string
target_in_TrainData = TrainData[target_value]
target_in_ValData = ValData[target_value]
#checking to make sure we have the right column selected
print(f'Target: {target_value}')
print(f'Target Train Data Check: {target_in_TrainData.head()}')
print(f'Target Val Data Check: {target_in_ValData.head()}')

#Features Values Setup
features_values = [col for col in TrainData.columns if col != target_value]
features_in_TrainData = TrainData[features_values]
features_in_ValData = ValData[features_values]
print(f'Features: {features_values}')
print(f'Features Train Data Check: {features_in_TrainData.head()}')
print(f'Features Val Data Check: {features_in_ValData.head()}')

   ID Gender Admit Type  Age at First Term                           Major  \
0   1      M         FR                 18                      Philosophy   
1   2      F        TRN                 20                         History   
2   3      F        NaN                 22                       Economics   
3   4      F        NaN                 21  English - Writing and Rhetoric   
4   5      M         FR                 18            Sport Administration   

   High School GPA  SAT Verbal  SAT Math  ACT Reading  ACT English  ...  \
0             3.49       690.0     630.0         25.0         28.0  ...   
1             2.97         NaN       NaN          NaN          NaN  ...   
2             3.05       360.0     690.0          NaN          NaN  ...   
3             3.36       510.0     560.0          NaN          NaN  ...   
4             3.64       720.0     640.0         29.0         32.0  ...   

   Attmpt Units Term 3  Passed Units Term 3  GPA Term 3  Attmpt Units Term 4  \


/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


Train Data Size: 564
Validation Data Size: 142
Target: GPA
Target Train Data Check: 155    3.157
210    3.239
260    3.094
424    3.284
539    3.774
Name: GPA, dtype: float64
Target Val Data Check: 478    3.991
81     3.835
77     3.335
208    3.436
319    3.473
Name: GPA, dtype: float64
Features: ['ID', 'Age at First Term', 'High School GPA', 'SAT Verbal', 'SAT Math', 'ACT Reading', 'ACT English', 'ACT Math', 'ACT Science', 'Attmpt Units Term 1', 'Passed Units Term 1', 'GPA Term 1', 'Attmpt Units Term 2', 'Passed Units Term 2', 'GPA Term 2', 'Attmpt Units Term 3', 'Passed Units Term 3', 'GPA Term 3', 'Attmpt Units Term 4', 'Passed Units Term 4', 'GPA Term 4', 'Attmpt Units Term 5', 'Passed Units Term 5', 'GPA Term 5', 'Gender_M', 'Admit Type_IFR', 'Admit Type_ITR', 'Admit Type_TRN', 'Major_Advertising', 'Major_Art', 'Major_Art History', 'Major_Biology -- BA', 'Major_Biology -- BS', 'Major_Business Administration', 'Major_Chemistry -- BA', 'Major_Chemistry -- BS', 'Major_Cinematic Arts

In [3]:
#Test Data Setup

import pandas as pd

file_path2 = '/kaggle/input/project-2-cosc-220-spring-2025/test.csv'
TestData = pd.read_csv(file_path2)

#checking to make sure the data was imported correctly
print(TestData.head())

#Changes the categorical variables to number format
#get_dummies = does one hot encoding on the values in the dataset
TestData = pd.get_dummies(TestData, columns=['Gender', 'Admit Type', 'Major'], drop_first=True)
#ChatGPT recommended adding this to check that the columns in ValData match the columns in TrainData
TestData = TestData.reindex(columns=TrainData.columns, fill_value=0)

#____________________________________________________________________________________________________________________
#EXPERIMENT: How to deal with zero values in the data
# 2 options: delete zero data columns, or replace with the mean. 
#I went with the mean option
TestData.fillna(TestData.mean(), inplace=True)
#____________________________________________________________________________________________________________________

#Features Values Setup
features_values2 = [col for col in TestData.columns]
features_in_TestData = TestData[features_values2]
print(f'Features: {features_values2}')
print(f'Features Train Data Check: {features_in_TestData.head()}')

     ID Gender Admit Type  Age at First Term                     Major  \
0  1001      M         FR                 18                Psychology   
1  1002      M        TRN                 22  Integrated Marketing Com   
2  1003      F         FR                 18  Integrated Marketing Com   
3  1004      F        TRN                 20  Integrated Marketing Com   
4  1005      M        TRN                 20                 Economics   

   High School GPA  SAT Verbal  SAT Math  ACT Reading  ACT English  ...  \
0             3.82         NaN       NaN         35.0         35.0  ...   
1             2.86       500.0     620.0          NaN          NaN  ...   
2             3.40         NaN       NaN         22.0         29.0  ...   
3             3.00       590.0     640.0          NaN          NaN  ...   
4             3.68         NaN       NaN          NaN          NaN  ...   

   GPA Term 2  Attmpt Units Term 3  Passed Units Term 3  GPA Term 3  \
0       3.772                    

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [4]:
#This is the model! 

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold #trying out cross validation...
from sklearn.linear_model import LassoCV

#TRAIN DATA SECTION
#Prepare Train Data
#features
X = features_in_TrainData
#target
y = target_in_TrainData
#Convert the data into numpy arrays for better processing?
X = np.array(X)
y = np.array(y)

#VALIDATIION DATA SECTION
#changing the variable names so they don't match with trainData
# Prepare Validation data
Xval = features_in_ValData
yval = target_in_ValData
#Convert the data into numpy arrays for better processing?
Xval = np.array(Xval)
yval = np.array(yval)

#TEST DATA SECTION
#Prepare Test data
#features
Xtest = features_in_TestData
#ChatGPT said to reindex TestData to get the column numbers to match 
Xtest = Xtest[features_in_TrainData.columns]  
Xtest = np.array(Xtest)  

#________________________________________________________________________________________________________________
#EXPERIMENT:Trying out LassoCV
#apparently LassoCV only runs with pandas dataframes, so I need to make a converted version of X
X_dataframe = pd.DataFrame(X)
lasso = LassoCV(cv=10, random_state=42).fit(X_dataframe, y)
important_features = X_dataframe.columns[lasso.coef_ != 0]
X_selected = X_dataframe[important_features]
Xtest_selected = pd.DataFrame(Xtest, columns=X_dataframe.columns)[important_features]
#supposedly will print what features are important to keep
#Well... the important features are in their numerical form, so idk which is which
print("\nImportant Features Selected by LassoCV:")
print(important_features.tolist())
#Let's try removing the unimportant variables accoreding to LassoCV
#for train data
X_selected = X_dataframe[important_features].to_numpy()
#for validation data
Xval_selected = pd.DataFrame(Xval, columns=X_dataframe.columns)[important_features].to_numpy()
#for test data
Xtest_selected = pd.DataFrame(Xtest, columns=X_dataframe.columns)[important_features].to_numpy()
#Train a new SVR model using only these features
svr = SVR(kernel='linear')
svr.fit(X_selected, y)
#Predict on test set
test_predictions = svr.predict(Xtest_selected)
#________________________________________________________________________________________________________________
#Pipeline for model 
procedure = make_pipeline(StandardScaler(), SVR(kernel='linear', C=0.1, epsilon=0.2))
#________________________________________________________________________________________________________________
#VALIDATION
#Training the model
pipeline = procedure.fit(X_selected, y)
#________________________________________________________________________________________________________________
#EXPERIMENT: Trying Cross Validation to improve accuracy?
#setting up KFold (you can experiment with the parameters here)
kfold = KFold(n_splits=20, shuffle=True, random_state=42)
#performing the cross validaiton on each fold
scores = -cross_val_score(pipeline, X_selected, y, cv=kfold, scoring='neg_mean_squared_error')
#printing the Cross Validation MSE 
print(f'Cross Validation MSE: {scores.mean():.4f} ±{scores.std():.4f}')
#________________________________________________________________________________________________________________

#Predict the target variable (GPA)
val_prediction = pipeline.predict(Xval_selected)
#MSE evaluation for Validation set
mse_val = mean_squared_error(yval, val_prediction)

#TEST
#Predict the target variable (GPA)
test_prediction = pipeline.predict(Xtest_selected)

#Correct Output of Prediction for competiton
#apparently I need to round to the nearest tenth for my predictions, whoops...
test_prediction_cut = np.round(test_prediction, 1)
output = pd.DataFrame({
    'ID': TestData['ID'],
    'GPA': np.round(test_prediction, 1)
})
output.to_csv('submission.csv', index=False)
print('YAY! The model worked right :)')
print('The submission.csv file was successfully saved!')


Important Features Selected by LassoCV:
[0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 18, 19, 20, 21, 23, 31, 33, 34, 67, 77]
Cross Validation MSE: 0.1729 ±0.1716
YAY! The model worked right :)
The submission.csv file was successfully saved!


In [5]:
#Model Evaluation
print('Here is your model\'s evaluation:')
print(f'Validation set MSE: {mse_val:.3f}')

Here is your model's evaluation:
Validation set MSE: 0.142


# The Baseline Model is complete! Time to experiment! :)

# Score Sheet!

**A way of keeping track how my model does validation vs test predictions**

Submission No.1
* Validation MSE: 0.133
* Test MSE: 0.150

*There's probably some overfitting that is happening*
*Although Dr. Scalzo said it is barely any, so it's looking good! :)*

GridSearchCV Experiment Results
* These parameters works best: {'svr__C': 0.1, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'}
* This is the best Validation MSE with those parameters: -0.17287687352722292

Submission No. 2 (based on GridSearchCV results)
* Validation set MSE: 0.132
* Test set MSE: 0.155

*The difference is not good enough for me to justify keeping the parameters from the GridSearchCV experiment*

# To Do List

OKAY so the project description says to create a regression model, so that is what I will make. 

**Part 1 is complete! We have a baseline model :)**

**Part 2: Experimenting**
* Try Recursive feature elemination with SVR?

**Important Note from Dr. Scalzo**
* The best way to prevent overfitting is to simplify the model
* Ex: If it's an SVR model, mess with the model's kernel or C value
* Ex: If its a decision trees model, lower the amount of decision trees


# The Graveyard

**All of the failed experiments so I can keep track of what methods didn't work**

* deleting feature values (it worked really well with all of the features included)
* Keeping all of the majors (not relevant enough to matter I guess, I ended up just one-hot encoding them and it worked well)
* replacing missing values with zero (mean replacement > zero replacement)
* RBF kernel in SVR
* Changing the C value in SVR
* GridSearchCV
* LassoCV alpha parameter (does it automatically)
* PCA (made model a lot worse)